# Advanced AI-Driven Search
This notebook is a companion to the article Advanced AI-Driven Search demonstrates use of advanced techniques that can be leveraged to improve the performance of traditional RAG architrecture. </br>
Specifically, it covers four approaches: **learned sprase retrieval**, **mixture of encoders**, **wormhole vectors** and **relevance feedback**. </br>
Each of these techniques is explained and visualized. On top of that, there is always resuable code block that you can leverage in your application.

In [1]:
%cd ../

/Users/jakubzovakfilevine/Desktop/advanced_ai_driven_search


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from typing import cast

from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from datasets.dataset_dict import Dataset
from qdrant_client import QdrantClient
from qdrant_client.models import models
from qdrant_client.http.models.models import QueryResponse
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from fastembed.postprocess import Muvera
from fastembed.sparse.sparse_embedding_base import SparseEmbedding

import numpy as np

from notebooks import (
    build_doc_vector,
    build_query_vector,
    display_retrieval_result,
)

## Dataset
For the demonstrations of the advanced concepts we will use modified version of the MS Macro dataset. <br>
Load the data from the Hugging Face dataset from [Zovi3/pa195_semestral_assignment](https://huggingface.co/datasets/Zovi3/pa195_semestral_assignment/upload/main). 

In [4]:
query_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_semestral_assignment",
    data_dir="query-all-MiniLM-L6-v2-100-filters-embedded-results"
)) 
query_dataset: Dataset = query_dataset_dict["train"]
query_dataset

Dataset({
    features: ['text', 'id', 'filters', 'embedding', 'multi_vector_embedding', 'result'],
    num_rows: 100
})

In [5]:
# Locally prepared corpus (dense + multi-vector + precomputed SPLADE + rating).
# Produced by notebooks/prepare_dataset.ipynb and saved under ./data/.
NUM_THOUSANDS = 1
DOCUMENTS_DATASET_NAME = (
    f"corpus-all-MiniLM-L6-v2-{NUM_THOUSANDS}K-multi-vector-splade-metadata"
)

documents_dataset_dict: DatasetDict = DatasetDict.load_from_disk(
    f"./data/{DOCUMENTS_DATASET_NAME}"
)

documents: Dataset = documents_dataset_dict["train"]
documents

Dataset({
    features: ['text', 'id', 'embedding', 'multi_vector_embedding', 'splade_indices', 'splade_values', 'rating'],
    num_rows: 1000
})

## Setup

Define constatnts to be used later.

In [6]:
COLLECTION_NAME = "ms_macro"
DENSE_VECTOR_FIELD = "dense"
BM25_VECTOR_FIELD = "bm25"
SPLADE_VECTOR_FIELD = "splade"
MULTI_VECTOR_FIELD = "multi"
MUVERA_VECTOR_FIELD = "muvera"
METADATA_AWARE_VECTOR_FIELD = "metadata_aware"

QUERY_EXAMPLE = "what is the study of virology"

Define models we are going to use throughout the notebook.

In [7]:
embedding_model = TextEmbedding('sentence-transformers/all-MiniLM-L6-v2')
embedding_model_size = 384

BM25_MODEL_NAME = "Qdrant/bm25"
bm25_model = SparseTextEmbedding(BM25_MODEL_NAME)

SPLADE_MODEL_NAME = "prithivida/Splade_PP_en_v1"
splade_model = SparseTextEmbedding(SPLADE_MODEL_NAME)

multi_vector_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
multi_vector_model_size = 128

# MUVERA turns the variable-length ColBERT multi-vector into a single fixed-size
# vector, enabling fast HNSW first-stage retrieval before ColBERT reranking.
muvera = Muvera.from_multivector_model(
    model=multi_vector_model, k_sim=6, dim_proj=32, r_reps=20
)

NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from /var/folders/26/rfvd55vx165d56kkxk2y3kr40000gn/T/fastembed_cache/models--Qdrant--Splade_PP_en_v1/snapshots/efcd182bc7eb351e81a9445752d4388c2bab500b/model.onnx failed:Load model /var/folders/26/rfvd55vx165d56kkxk2y3kr40000gn/T/fastembed_cache/models--Qdrant--Splade_PP_en_v1/snapshots/efcd182bc7eb351e81a9445752d4388c2bab500b/model.onnx failed. File doesn't exist

Initiate the Qdrant in memory instance.

In [ ]:
client = QdrantClient(":memory:")

Create Qdrant collection to use for demonstration of different search types. Specifics of this code will be explained in later sections. 

In [ ]:
try:
    client.delete_collection(COLLECTION_NAME)
except: 
    print(f"Collection {COLLECTION_NAME} does not exist. Creating new one.")
    
collection_created = client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        DENSE_VECTOR_FIELD: models.VectorParams(
            size=embedding_model_size,
            distance=models.Distance.COSINE
        ),
        MULTI_VECTOR_FIELD: models.VectorParams(
            distance=models.Distance.COSINE,
            size=multi_vector_model_size,
            multivector_config=models.MultiVectorConfig( # Configure max-sim operator
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0), # Disable HNSW index reranking
        ),
        MUVERA_VECTOR_FIELD: models.VectorParams(
            size=muvera.embedding_size, # Fixed-size MUVERA encoding of the multi-vector
            distance=models.Distance.COSINE,
        ),
        METADATA_AWARE_VECTOR_FIELD: models.VectorParams(
            size=embedding_model_size + 3, # dense embedding + 3D rating encoding
            distance=models.Distance.COSINE,
        ),
    },
    sparse_vectors_config={
        BM25_VECTOR_FIELD: models.SparseVectorParams(modifier=models.Modifier.IDF),
        SPLADE_VECTOR_FIELD: models.SparseVectorParams()
    }
)

if collection_created:
    print(f"Created collection '{COLLECTION_NAME}'.")

rating_weight = 0.5
points = [
    models.PointStruct(
        id=doc["id"],
        vector={
            DENSE_VECTOR_FIELD: doc["embedding"],
            MULTI_VECTOR_FIELD: doc["multi_vector_embedding"],
            MUVERA_VECTOR_FIELD: muvera.process_document(
                np.array(doc["multi_vector_embedding"])
            ).tolist(),
            METADATA_AWARE_VECTOR_FIELD: build_doc_vector(
                doc["embedding"], doc["rating"], rating_weight=rating_weight
            ).tolist(),
            BM25_VECTOR_FIELD: models.Document(
                    text=doc["text"], model=BM25_MODEL_NAME

            ),
            SPLADE_VECTOR_FIELD: models.SparseVector(
                    indices=doc["splade_indices"], values=doc["splade_values"]
            ),
        },
        payload={"text": doc["text"], "rating": doc["rating"]},
    )
    for doc in documents
]
client.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=128)
print(f"Collection info: {client.get_collection(COLLECTION_NAME).points_count} points in collection")

## Basic Retrieval Augmented Generation (RAG)
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        RAG is a standard design pattern used in almost all AI applications that require custom knowledge base.
        There are two main components in RAG: retrieval and generation. In this notebook we solely focus on retrieval.
      </p>
      <p>
        A RAG pipeline is visualized in Figure 1. It uses a <strong>vector database</strong> for storage and retrieval of embedded documents,
        includes a re-ranking step, and then passes the retrieved documents to an LLM to generate the final answer.
      </p>
      <p>
The retrieval is done based on similarity which is defined by the distance function, in practice it is almost always cosine distance:

$$
\mathrm{d}(x,y) = 1 - \cos(x,y) = 1 - \frac{x \cdot y}{\|x\| \|y\|}
$$

In this notebook we use [Qdrant](https://github.com/qdrant/qdrant) as our vector database. It offers many advanced features like relevance feedback, multi-vector support..
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="https://raw.githubusercontent.com/Zovi343/wormhole_vectors/main/images/rag_pipeline.png" alt="Diagram of a RAG pipeline" width="500" />
      <p><b>Figure 1. RAG pipeline</b></p>
    </td>
  </tr>
</table>

#### Vector Search Query

In [ ]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_dense_embedding,
        using="dense",
        limit=5
)

display_retrieval_result(retrieval_results)

## Sparse Retrieval
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Hybrid-search is a natural step for improving basic RAG pipeline.
      </p>
      <p>
        BM25 algorithm is a default approach when it comes to the sparse retrieval. <br>
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/hybrid_search.png" alt="Diagram of a Hybrid Retrieval" width="450" />
      <p><b>RAG pipeline</b></p>
    </td>
  </tr>
</table>

### Hybrid Search Query

In [ ]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

sparse_limit = 10
dense_limit = 10
prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
                query=models.Document(
                        text=QUERY_EXAMPLE,
                        model=BM25_MODEL_NAME
                ),
                using=BM25_VECTOR_FIELD,
                limit=sparse_limit,
        ),
        models.Prefetch(
                query=query_dense_embedding,
                using=DENSE_VECTOR_FIELD,
                limit=dense_limit,
        ),
]

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_sparse_and_dense_search,
        query=models.RrfQuery(rrf=models.Rrf(k=60)),
        limit=5
)

display_retrieval_result(retrieval_results)

## Learned Sparse Retrieval
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Hybrid-search is a natural step for improving basic RAG pipeline.
      </p>
      <p>
        TODO
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/hybrid_search.png" alt="Diagram of a Hybrid Retrieval" width="300" />
      <p><b>Learned Sparse Retrieval</b></p>
    </td>
  </tr>
</table>

#### Splade
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Splade is lorem ipsum lorem ipsum
      </p>
      <p>
        BM25 model often stragles with synonyms, homnonyms and other subtle relations in the syntax. <br>
        Splade model addresses by performing query espansion and also enrichnig sparse representation with semantic information.
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/splade.png" alt="Diagram of a Hybrid Retrieval" width="250" />
      <p><b>SPLADE</b></p>
    </td>
  </tr>
</table>

### Hybrid Search Query With Splade

In [ ]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

sparse_limit = 10
dense_limit = 10
prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
                query=models.Document(
                        text=QUERY_EXAMPLE,
                        model=SPLADE_MODEL_NAME
                ),
                using=SPLADE_VECTOR_FIELD,
                limit=sparse_limit,
        ),
        models.Prefetch(
                query=query_dense_embedding,
                using=DENSE_VECTOR_FIELD,
                limit=dense_limit,
        ),
]

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_sparse_and_dense_search,
        query=models.RrfQuery(rrf=models.Rrf(k=60)),
        limit=5
)

display_retrieval_result(retrieval_results)

## Metadata Aware Vector Search

### Metadata Boosting
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Metadata boosting is lorem ipsum lorem ipsum
      </p>
      <p>
        TODO
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/metadata_boosting.png" alt="Diagram of a Hybrid Retrieval" width="400" />
      <p><b>Metadata Boosting</b></p>
    </td>
  </tr>
</table>

In [ ]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

dense_limit = 10
prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
                query=query_dense_embedding,
                using=DENSE_VECTOR_FIELD,
                limit=dense_limit,
        ),
]


rating_boost_weight = 0.05
retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_sparse_and_dense_search,
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=[
                "$score",
                # Boost each document by its rating (1-5) scaled by the weight
                models.MultExpression(mult=[rating_boost_weight, "rating"]),
            ]
        )),
        limit=5
)

display_retrieval_result(retrieval_results)

### Mixture of Encoders
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Splade is lorem ipsum lorem ipsum
      </p>
      <p>
        TODO
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/mixture_of_encoders.png" alt="Diagram of a Hybrid Retrieval" width="400" />
      <p><b>Mixture of Encoders</b></p>
    </td>
  </tr>
</table>

In [ ]:
query_metadata_aware = build_query_vector(
    query_dense_embedding, rating_weight=rating_weight
).tolist()

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_metadata_aware,
        using=METADATA_AWARE_VECTOR_FIELD,
        limit=5
)

display_retrieval_result(retrieval_results)

### Multi-Vector Model

<table>
  <tr>
    <td style="width: 50%; vertical-align: top;">
      <p>
        As mentioned above, we will implement re-ranking with multi-vector late interaction embeddings based on
        <a href="https://dl.acm.org/doi/epdf/10.1145/3397271.3401075">ColBERT</a>.
      </p>
      <p>
        In traditional encoder models such as BERT, the final vector is obtained by pooling over token embeddings. <br>
        In the late interaction approach, we skip pooling and instead use token-level embeddings to represent documents and queries. 
      </p>
      <p>
        This yields a much more expressive representation, usually at the cost of higher memory usage and query latency. <br>
        That is why late interaction is commonly used as a re-ranking step instead of the main retrieval stage.
      </p>
      <p>
      Late interaction can be leveraged as a re-ranking step with the use of <> the following distance function, called the MaxSim operator:

$$
\text{MaxSim}(q, D) = \sum_{i \in q} \max_{j \in D} \; \text{sim}(q_i, d_j)
$$

The MaxSim operator computes cosine similarity between each query token and each document token and then takes the maximum similarity for each query token. <br> It is visualized in **Figure 3**.
</p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="https://raw.githubusercontent.com/Zovi343/wormhole_vectors/main/images/max_sim.png" alt="Late Interaction With MaxSim Operator" width="500" />
      <p><b>Late Interaction With MaxSim Operator</b></p>
    </td>
  </tr>
</table>

### Multi-vector Search
It is general good practice to include reranking model in the IR system. <br>
Reranking uses stronger model to select the most relevant documents from the initial retrieval. <br>
You will implement reranking with multi-vector late interaction embedding ColBERT.

In [ ]:
# Multi-vector search: fast MUVERA retrieval first, then re-rank the
# candidates with the ColBERT late-interaction (MaxSim) multi-vector.
query_multi_vector = list(multi_vector_model.query_embed(QUERY_EXAMPLE))[0]
query_muvera = muvera.process_query(query_multi_vector)

rerank_prefetch_limit = 20
retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=models.Prefetch(
                query=query_muvera.tolist(),
                using=MUVERA_VECTOR_FIELD,
                limit=rerank_prefetch_limit, # Stage 1: fast MUVERA retrieval
        ),
        query=query_multi_vector.tolist(), # Stage 2: ColBERT MaxSim rerank
        using=MULTI_VECTOR_FIELD,
        limit=5
)

display_retrieval_result(retrieval_results)

### Wormhole Vectors

<table>
  <tr>
    <td style="width: 50%; vertical-align: top;">
      <p>
        Standard hybrid search treats dense (semantic) and sparse (lexical) vector spaces as
        disjoint silos and merely fuses their result lists (e.g. with RRF). <b>Wormhole vectors</b>,
        an idea by Trey Grainger described in
        <a href="https://aiven.io/blog/beyond-hybrid-search-traversing-vector-spaces-with-wormhole-vectors">this Aiven article</a>,
        instead use the shared <i>document set</i> as a bridge to <i>traverse</i> between spaces.
      </p>
      <p>
        We demonstrate both directions:
      </p>
      <ul>
        <li>
          <b>Dense &rarr; Sparse:</b> run a dense query to obtain a foreground set of documents,
          then find terms that are statistically over-represented in the foreground vs. the whole
          corpus (a <code>significant_terms</code>-style analysis) combined with positional weighting
          of top-ranked docs. These "explanatory" keywords are hopped back into a lexical BM25 query.
        </li>
        <li>
          <b>Sparse &rarr; Dense:</b> run a lexical (BM25) query, mean-pool the retrieved documents'
          dense embeddings into a centroid "wormhole vector", then hop into dense space to surface
          semantically related documents that share no keywords.
        </li>
      </ul>
      <p>
        This bridges the vocabulary gap (the "zero result" problem) and adds explainability by
        mapping dense regions back to human-readable keywords.
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/wormhole_vectors.png" alt="Wormhole Vectors" width="500" />
      <p><b>Wormhole Vectors</b></p>
    </td>
  </tr>
</table>

In [ ]:
from collections import defaultdict
from gensim.parsing.preprocessing import STOPWORDS


def tokenize(text: str) -> list[str]:
    tokens = (t.strip(".,!?;:()[]{}\"'") for t in text.lower().split())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


# Background term document-frequencies over the whole corpus (computed once).
background_df: dict[str, int] = defaultdict(int)
for _doc in documents:
    for _term in set(tokenize(_doc["text"])):
        background_df[_term] += 1
background_total = len(documents)


def significant_terms(foreground_texts, keyword_size=10, min_fg_count=2):
    # Qdrant has no significant_terms aggregation, so replicate a JLH-style
    # significance measure comparing foreground vs. background probabilities.
    fg_df: dict[str, int] = defaultdict(int)
    for text in foreground_texts:
        for term in set(tokenize(text)):
            fg_df[term] += 1
    fg_total = max(len(foreground_texts), 1)
    out = []
    for term, fg in fg_df.items():
        if fg < min_fg_count:
            continue
        bg = background_df.get(term, fg)
        p_fg, p_bg = fg / fg_total, bg / background_total
        score = (p_fg - p_bg) * (p_fg / p_bg)
        out.append(
            {
                "term": term,
                "score": score,
                "foreground_count": fg,
                "background_count": bg,
            }
        )
    out.sort(key=lambda x: x["score"], reverse=True)
    return out[:keyword_size]


def positional_weights(foreground_texts, top_n=5):
    # Terms from higher-ranked documents get more weight (1/sqrt(rank) decay).
    weights: dict[str, float] = defaultdict(float)
    for rank, text in enumerate(foreground_texts[:top_n], 1):
        w = 1.0 / (rank ** 0.5)
        for term in tokenize(text):
            weights[term] += w
    return dict(weights)


def combine_keyword_scores(stat_kw, pos_w, stat_weight=0.4, pos_weight=0.6):
    norm_pos = {}
    if pos_w:
        mx = max(pos_w.values())
        norm_pos = {k: v / mx for k, v in pos_w.items()} if mx > 0 else {}
    norm_stat = {}
    if stat_kw:
        scores = [k["score"] for k in stat_kw]
        lo, hi = min(scores), max(scores)
        rng = hi - lo if hi > lo else 1.0
        norm_stat = {k["term"]: (k["score"] - lo) / rng for k in stat_kw}
    combined = {}
    for k in stat_kw:
        t = k["term"]
        combined[t] = {
            "term": t,
            "statistical_score": k["score"],
            "positional_score": norm_pos.get(t, 0.0),
            "combined_score": stat_weight * norm_stat.get(t, 0.0)
            + pos_weight * norm_pos.get(t, 0.0),
            "foreground_count": k["foreground_count"],
            "background_count": k["background_count"],
        }
    # Inject strong positional-only terms that were not statistically significant.
    for t, ps in norm_pos.items():
        if t not in combined and ps > 0.3:
            combined[t] = {
                "term": t,
                "statistical_score": 0.0,
                "positional_score": ps,
                "combined_score": pos_weight * ps,
                "foreground_count": 0,
                "background_count": 0,
            }
    return sorted(combined.values(), key=lambda x: x["combined_score"], reverse=True)


def wormhole_dense_to_sparse(query_text, k=20, keyword_size=10, num_terms=5):
    # Dense query -> foreground docs -> explanatory keywords -> BM25 lexical hop.
    q_dense = list(embedding_model.query_embed(query_text))[0].tolist()
    fg = client.query_points(
        collection_name=COLLECTION_NAME,
        query=q_dense,
        using=DENSE_VECTOR_FIELD,
        limit=k,
        with_payload=True,
    )
    fg_texts = [p.payload["text"] for p in fg.points]

    wormhole_kw = combine_keyword_scores(
        significant_terms(fg_texts, keyword_size), positional_weights(fg_texts)
    )
    top_terms = [kw["term"] for kw in wormhole_kw[:num_terms]]

    sparse = client.query_points(
        collection_name=COLLECTION_NAME,
        query=models.Document(text=" ".join(top_terms), model=BM25_MODEL_NAME),
        using=BM25_VECTOR_FIELD,
        limit=k,
        with_payload=True,
    )

    dense_ids = {p.id for p in fg.points}
    sparse_ids = {p.id for p in sparse.points}
    overlap = len(dense_ids & sparse_ids)

    print(f"[Dense -> Sparse] Query: {query_text!r}")
    print(f"Wormhole keywords: {top_terms}")
    print(
        f"Dense/sparse overlap: {overlap}/{len(dense_ids)} "
        f"({overlap / max(len(dense_ids), 1):.0%})\n"
    )
    print("Top wormhole keywords (combined score):")
    for kw in wormhole_kw[:keyword_size]:
        print(
            f"  {kw['term']:<15} {kw['combined_score']:.3f} "
            f"(stat={kw['statistical_score']:.3f}, pos={kw['positional_score']:.3f})"
        )
    print("\nSparse (BM25) results from wormhole keywords:")
    display_retrieval_result(sparse)
    return wormhole_kw


def wormhole_sparse_to_dense(query_text, k=20):
    # Lexical query -> mean-pool foreground dense vectors -> dense semantic hop.
    fg = client.query_points(
        collection_name=COLLECTION_NAME,
        query=models.Document(text=query_text, model=BM25_MODEL_NAME),
        using=BM25_VECTOR_FIELD,
        limit=k,
        with_payload=True,
        with_vectors=[DENSE_VECTOR_FIELD],
    )
    dense_vecs = np.array([p.vector[DENSE_VECTOR_FIELD] for p in fg.points])
    centroid = dense_vecs.mean(axis=0)  # mean pooling -> centroid wormhole vector

    dense = client.query_points(
        collection_name=COLLECTION_NAME,
        query=centroid.tolist(),
        using=DENSE_VECTOR_FIELD,
        limit=k,
        with_payload=True,
    )

    sparse_ids = {p.id for p in fg.points}
    dense_ids = {p.id for p in dense.points}
    overlap = len(sparse_ids & dense_ids)

    print(f"[Sparse -> Dense] Query: {query_text!r}")
    print(
        f"Sparse/dense overlap: {overlap}/{len(sparse_ids)} "
        f"({overlap / max(len(sparse_ids), 1):.0%})\n"
    )
    print("Dense results from mean-pooled centroid:")
    display_retrieval_result(dense)
    return centroid


_ = wormhole_dense_to_sparse(QUERY_EXAMPLE, k=20, keyword_size=10)
print("\n" + "=" * 65 + "\n")
_ = wormhole_sparse_to_dense(QUERY_EXAMPLE, k=20)